# 📊 Module 1.2 — Asset Classes & Benchmarks
### *From Theory to Real-World Trading with Python*

---

**Course:** Capital Markets Recap — Level 1: Foundations  
**Prerequisites:** Module 1.1 (Financial Returns)  
**Tools:** `yfinance`, `pandas`, `numpy`, `matplotlib`

---

## What This Module Covers

In your lecture notes, returns were contextualized by comparing different **asset classes** — T-bills, Treasury bonds, corporate bonds, large-cap stocks (S&P 500), and small-cap stocks. This is one of the foundational ideas in finance: **different asset classes have different risk-return profiles**, and understanding them is essential before making any investment decision.

This module covers:
1. **The main asset classes** — what they are and how they work
2. **The risk-free rate** — T-bills as the baseline
3. **Equity benchmarks** — S&P 500, Nasdaq, and how to use them
4. **The empirical risk-return tradeoff** — does higher risk actually deliver higher return?
5. **Correlations between asset classes** — the foundation of diversification
6. **Drawdown analysis** — understanding the pain behind the return numbers

---
## 0. Setup

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['font.size'] = 11

TRADING_DAYS = 252
START = '2010-01-01'
END   = '2024-12-31'

print("Setup complete ✅")

---
## 1. The Main Asset Classes

### Theory

Your lecture notes described the main investable asset classes and their risk-return ladder — from safest to riskiest:

| Asset Class | What it is | Risk level |
|---|---|---|
| **T-bills** | Short-term US government debt (<1 year) | Near zero — *the risk-free rate* |
| **Treasury Bonds** | Long-term US government debt (10–30 yr) | Low — interest rate risk only |
| **Corporate Bonds** | Debt issued by companies | Medium — adds credit risk |
| **Large-cap Equities** (S&P 500) | Shares in the 500 largest US companies | High — market risk |
| **Small-cap Equities** | Shares in smaller companies | Higher — + liquidity & idiosyncratic risk |

Your notes quoted from the classic Ibbotson chart: *'Invested in 1925'* — illustrating how $1 invested in small stocks compounded to far more than $1 in T-bills, but with much greater volatility along the way.

### The ETF proxies we'll use
Since we can't easily download 100-year daily data via yfinance, we'll use **ETFs** as liquid, real-world proxies:

| ETF | Proxy for | Description |
|---|---|---|
| `SHY` | T-bills / Short-term Treasuries | iShares 1-3 Year Treasury Bond ETF |
| `IEF` | Intermediate Treasuries | iShares 7-10 Year Treasury Bond ETF |
| `TLT` | Long-term Treasuries | iShares 20+ Year Treasury Bond ETF |
| `LQD` | Investment-grade corporate bonds | iShares iBoxx $ Investment Grade Corp Bond ETF |
| `HYG` | High-yield (junk) corporate bonds | iShares iBoxx $ High Yield Corporate Bond ETF |
| `SPY` | S&P 500 (large-cap equities) | SPDR S&P 500 ETF Trust |
| `IWM` | Russell 2000 (small-cap equities) | iShares Russell 2000 ETF |
| `QQQ` | Nasdaq 100 (tech-heavy) | Invesco QQQ Trust |

In [ ]:
# ── 1a. Download all asset class ETFs ────────────────────────────────────────

ASSETS = {
    'SHY': 'Short-term Treasuries',
    'IEF': 'Mid-term Treasuries',
    'TLT': 'Long-term Treasuries',
    'LQD': 'Investment Grade Corp Bonds',
    'HYG': 'High Yield Corp Bonds',
    'SPY': 'S&P 500 (Large Cap)',
    'IWM': 'Russell 2000 (Small Cap)',
    'QQQ': 'Nasdaq 100 (Tech)',
}

prices = yf.download(
    list(ASSETS.keys()),
    start=START, end=END,
    auto_adjust=True, progress=False
)['Close']

# Rename columns to descriptive names
prices.columns = [ASSETS[t] for t in prices.columns]

# Drop any days with all NaN (e.g. before some ETFs launched)
prices = prices.dropna(how='all')

print(f"Downloaded {len(prices)} trading days ({START[:4]}–{END[:4]})")
print(f"Assets     : {list(prices.columns)}")
print(f"Date range : {prices.index[0].date()} → {prices.index[-1].date()}")

---
## 2. The Risk-Free Rate — T-Bills as the Baseline

### Theory

The **risk-free rate** ($r_f$) is the return an investor can earn with **zero risk**. In practice, this is the yield on short-term US Treasury bills (T-bills), because:

- The US government is considered unable to default on short-term debt (it can always print dollars)
- Short maturity (< 1 year) means minimal **interest rate risk** — if rates rise, you get your money back quickly and reinvest at higher rates

**Why does the risk-free rate matter?**

It's the **floor** of the return ladder. Every other investment must offer a return *above* $r_f$ to compensate investors for taking on risk. This excess return is called the **risk premium**:

$$
\text{Risk Premium} = E[R_{asset}] - r_f
$$

You'll use $r_f$ constantly in trading:
- As the denominator's reference in the **Sharpe Ratio** (Module 1.3)
- As the baseline in the **CAPM** formula (Module 2.3): $E[R_i] = r_f + \beta(E[R_m] - r_f)$
- To evaluate whether a strategy is actually adding value beyond a risk-free parking

> 💡 **yfinance tip:** The 13-week T-bill yield (ticker `^IRX`) gives you the current annualized risk-free rate directly from CBOE data.

In [ ]:
# ── 2a. Download 13-week T-bill yield (risk-free rate proxy) ─────────────────
# ^IRX = 13-week T-bill yield quoted as annualized %, from CBOE

tbill = yf.download('^IRX', start=START, end=END, progress=False)['Close']
tbill = tbill / 100  # convert from percentage to decimal
tbill.name = '13W T-Bill Yield'

# Daily equivalent of the annualized yield
rf_daily = tbill / TRADING_DAYS

# Current and average
rf_current = tbill.iloc[-1]
rf_avg     = tbill.mean()

print(f"Current 13W T-bill yield  : {rf_current*100:.2f}%  (annualized)")
print(f"Average over {START[:4]}–{END[:4]} : {rf_avg*100:.2f}%  (annualized)")
print(f"Daily equivalent (avg)    : {rf_daily.mean()*100:.4f}%")

In [ ]:
# ── 2b. Plot the history of the risk-free rate ───────────────────────────────
plt.figure(figsize=(13, 4))
plt.plot(tbill.index, tbill * 100, color='#2c3e50', linewidth=1.5)
plt.fill_between(tbill.index, 0, tbill * 100, alpha=0.15, color='#2c3e50')
plt.axhline(rf_avg * 100, color='orange', linestyle='--', linewidth=1.5,
            label=f'Period average: {rf_avg*100:.2f}%')

# Annotate key macro events
events = {
    '2020-03': ('COVID\nZIRP', 'red'),
    '2022-03': ('Fed\nhike cycle', 'darkred'),
}
for date_str, (label, color) in events.items():
    dt = pd.to_datetime(date_str)
    if dt in tbill.index or tbill.index.searchsorted(dt) < len(tbill):
        idx = tbill.index.searchsorted(dt)
        plt.axvline(tbill.index[idx], color=color, linestyle=':', alpha=0.7)
        plt.text(tbill.index[idx], tbill.max()*100*0.8, label,
                 fontsize=8, color=color, ha='center')

plt.title('13-Week T-Bill Yield — The Risk-Free Rate Over Time', fontweight='bold')
plt.ylabel('Annualized Yield (%)')
plt.legend()
plt.tight_layout()
plt.show()

print("\n📊 Key macro context:")
print("  - 2010–2015: Near-zero rates (ZIRP) following the 2008 financial crisis")
print("  - 2020: Rates slashed to 0% during COVID")
print("  - 2022–2024: Fastest rate hiking cycle in 40 years (inflation response)")
print("  These shifts have enormous implications for ALL asset class valuations.")

---
## 3. Equity Benchmarks — What They Are and How Traders Use Them

### Theory

A **benchmark** is a reference portfolio against which performance is measured. For equity traders, the most important benchmarks are:

**S&P 500 (SPY)** — the gold standard benchmark. Tracks the 500 largest US companies by market capitalization, covering ~80% of US equity market value. When someone says "the market", they usually mean the S&P 500.

**Nasdaq 100 (QQQ)** — the 100 largest non-financial companies listed on Nasdaq. Heavily weighted toward technology (Apple, Microsoft, Nvidia, Amazon). Higher growth potential but also higher volatility.

**Russell 2000 (IWM)** — tracks 2,000 small-cap US companies. Your notes describe small-cap stocks as having *"menor capitalización de mercado, mayor probabilidad de quiebra, pero así es el retorno"* — lower market cap, higher bankruptcy risk, but compensated with higher expected returns.

### The market cap weighting effect
Modern indices are **market-cap weighted** — larger companies have larger influence. This means the S&P 500 is effectively a momentum vehicle: companies that have already grown large get even larger weights.

> 💡 **Trading implication:** When you buy SPY, you're not buying 500 equal bets. The top 10 holdings can represent 30–35% of the entire index. Always check concentration.

In [ ]:
# ── 3a. Normalized equity benchmark comparison ───────────────────────────────
equity_cols = ['S&P 500 (Large Cap)', 'Russell 2000 (Small Cap)', 'Nasdaq 100 (Tech)']
equity_prices = prices[equity_cols].dropna()

normalized = equity_prices / equity_prices.iloc[0] * 100

colors_eq = ['#2980b9', '#e67e22', '#27ae60']

plt.figure(figsize=(13, 6))
for col, color in zip(normalized.columns, colors_eq):
    final_val = normalized[col].iloc[-1]
    plt.plot(normalized.index, normalized[col],
             label=f"{col} ({final_val:.0f})", color=color, linewidth=2)

plt.axhline(100, color='black', linewidth=0.7, linestyle='--', alpha=0.5)
plt.title('Equity Benchmarks — Normalized to 100 at Start', fontweight='bold')
plt.ylabel('Index Value (Start = 100)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 3b. Rolling 1-year return comparison across equity benchmarks ────────────
eq_returns = equity_prices.pct_change().dropna()
rolling_1yr = (1 + eq_returns).rolling(TRADING_DAYS).apply(np.prod, raw=True) - 1

fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
for ax, col, color in zip(axes, equity_cols, colors_eq):
    roll = rolling_1yr[col].dropna()
    ax.plot(roll.index, roll * 100, color=color, linewidth=1.5)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.fill_between(roll.index, 0, roll * 100,
                    where=(roll > 0), alpha=0.2, color='green')
    ax.fill_between(roll.index, 0, roll * 100,
                    where=(roll < 0), alpha=0.2, color='red')
    ax.set_title(col, fontweight='bold', fontsize=10)
    ax.set_ylabel('1-Yr Return (%)')

plt.suptitle('Rolling 1-Year Returns by Equity Benchmark', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

---
## 4. The Empirical Risk-Return Tradeoff

### Theory

The central hypothesis of classical finance is:

> **Higher risk should be compensated by higher expected return.**

Your notes put it concisely: *"Small Stocks: menor capitalización de mercado, mayor probabilidad de quiebra, pero así es el retorno"* — the higher risk of small caps should translate into higher long-run return.

We can visualize this empirically by plotting **annualized return** against **annualized volatility** for each asset class. In a well-functioning market, we'd expect a roughly upward-sloping relationship.

$$
\text{Annualized Return} \approx f(\text{Annualized Volatility})
$$

### The Sharpe Ratio — risk-adjusted return

The **Sharpe Ratio** measures how much *excess return* you earn per unit of *risk taken*:

$$
\text{Sharpe Ratio} = \frac{E[R_p] - r_f}{\sigma_p}
$$

Where:
- $E[R_p]$ = expected (average) portfolio return
- $r_f$ = risk-free rate
- $\sigma_p$ = portfolio standard deviation (annualized)

A Sharpe Ratio of:
- **< 0** → you'd have been better off in T-bills
- **0–0.5** → subpar but positive
- **0.5–1.0** → acceptable
- **> 1.0** → good; **> 2.0** → exceptional (and likely won't persist)

> 💡 Your lecture notes introduced the Sharpe Ratio as a *"desempeño de inversiones ajustado por el riesgo"* (risk-adjusted performance metric). It will be fundamental in portfolio construction in Level 2.

In [ ]:
# ── 4a. Compute risk-return statistics for all asset classes ─────────────────
all_returns = prices.pct_change().dropna()

# Use average annualized T-bill yield as risk-free rate
# Align tbill to the same index as all_returns
rf_aligned = tbill.reindex(all_returns.index, method='ffill').fillna(method='bfill')
rf_annual  = rf_aligned.mean()

stats = pd.DataFrame(index=prices.columns)

n_yrs = len(all_returns) / TRADING_DAYS
stats['CAGR (%)']       = ((1 + all_returns).prod() ** (1/n_yrs) - 1) * 100
stats['Volatility (%)'] = all_returns.std() * np.sqrt(TRADING_DAYS) * 100
stats['Sharpe Ratio']   = (stats['CAGR (%)'] / 100 - rf_annual) / (stats['Volatility (%)'] / 100)
stats['Max Daily Loss (%)'] = all_returns.min() * 100
stats['Max Daily Gain (%)'] = all_returns.max() * 100

# Sort by volatility (risk ladder)
stats = stats.sort_values('Volatility (%)')

print(f"Risk-Free Rate used: {rf_annual*100:.2f}% (period average)\n")
print(stats.round(3).to_string())

In [ ]:
# ── 4b. Risk-Return scatter plot — the empirical frontier ────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Group colors by asset type
asset_colors = {
    'Short-term Treasuries':       '#1abc9c',
    'Mid-term Treasuries':         '#16a085',
    'Long-term Treasuries':        '#0e6655',
    'Investment Grade Corp Bonds': '#f39c12',
    'High Yield Corp Bonds':       '#e67e22',
    'S&P 500 (Large Cap)':         '#2980b9',
    'Russell 2000 (Small Cap)':    '#8e44ad',
    'Nasdaq 100 (Tech)':           '#c0392b',
}

# Left: Risk vs Return
for asset in stats.index:
    x = stats.loc[asset, 'Volatility (%)']
    y = stats.loc[asset, 'CAGR (%)']
    color = asset_colors.get(asset, 'gray')
    axes[0].scatter(x, y, color=color, s=120, zorder=5)
    axes[0].annotate(asset.replace(' ', '\n'), (x, y),
                     textcoords='offset points', xytext=(6, 3),
                     fontsize=7.5, color=color)

axes[0].axhline(rf_annual * 100, color='gray', linestyle='--', linewidth=1,
                label=f'Risk-free rate ({rf_annual*100:.1f}%)')
axes[0].set_xlabel('Annualized Volatility (%)')
axes[0].set_ylabel('CAGR (%)')
axes[0].set_title('Risk-Return Tradeoff by Asset Class', fontweight='bold')
axes[0].legend(fontsize=9)

# Right: Sharpe Ratio bar chart
sharpe_sorted = stats['Sharpe Ratio'].sort_values()
bar_colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in sharpe_sorted]
axes[1].barh(sharpe_sorted.index, sharpe_sorted.values, color=bar_colors, alpha=0.8)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Sharpe Ratio')
axes[1].set_title('Sharpe Ratio by Asset Class\n(risk-adjusted return)', fontweight='bold')
# Shorten labels for readability
axes[1].set_yticklabels([l.get_text().replace(' (', '\n(') for l in axes[1].get_yticklabels()],
                         fontsize=8)

plt.tight_layout()
plt.show()

print("\n📊 Key observations:")
print("  - Equities generally sit in the upper-right (high risk, high return)")
print("  - Bonds cluster in the lower-left (low risk, lower return)")
print("  - The Sharpe Ratio tells you which assets gave the BEST return PER UNIT of risk")
print("  - A negative Sharpe means you earned LESS than T-bills — after adjusting for risk")

---
## 5. Correlation Between Asset Classes

### Theory

The **correlation coefficient** $\rho$ between two assets measures how their returns move together:

$$
\rho_{A,B} = \frac{Cov(R_A, R_B)}{\sigma_A \cdot \sigma_B} \in [-1, +1]
$$

- $\rho = +1$: perfect positive correlation — they always move together
- $\rho = 0$: no linear relationship
- $\rho = -1$: perfect negative correlation — when one rises, the other falls

This is the mathematical engine behind **diversification** (covered in depth in Module 2.1). For now, the key insight:

> If two assets are not perfectly correlated ($\rho < 1$), **combining them reduces portfolio risk** without necessarily reducing expected return.

### The flight-to-quality phenomenon
One of the most important empirical patterns in markets: during equity market crashes, investors sell stocks and buy government bonds. This causes **stocks and government bonds to become negatively correlated** during crises — making bonds a natural hedge for equity portfolios.

> ⚠️ **Important caveat (2022 lesson):** The negative stock-bond correlation is not a law of nature. In 2022, high inflation caused both stocks AND bonds to fall simultaneously — because rising interest rates hurt both asset classes. Correlations are **time-varying**.

In [ ]:
# ── 5a. Full-period correlation matrix ───────────────────────────────────────
import matplotlib.colors as mcolors

corr = all_returns.corr()

fig, ax = plt.subplots(figsize=(10, 8))

# Use a diverging colormap: red = positive corr, blue = negative
im = ax.imshow(corr, cmap='RdYlGn_r', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Correlation')

# Add correlation values as text
for i in range(len(corr)):
    for j in range(len(corr)):
        val = corr.iloc[i, j]
        txt_color = 'white' if abs(val) > 0.6 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=8, color=txt_color, fontweight='bold')

# Axis labels — short versions
short_names = [c.replace(' (', '\n(').replace('Investment Grade ', 'IG ')
               .replace('High Yield ', 'HY ') for c in corr.columns]
ax.set_xticks(range(len(corr)))
ax.set_yticks(range(len(corr)))
ax.set_xticklabels(short_names, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(short_names, fontsize=8)
ax.set_title(f'Asset Class Return Correlations ({START[:4]}–{END[:4]})',
             fontweight='bold', pad=15)

plt.tight_layout()
plt.show()

In [ ]:
# ── 5b. Rolling correlation — stock/bond relationship changes over time ───────
# Key pair: S&P 500 vs Long-term Treasuries
# This is the most important diversification relationship in traditional portfolios

spy_ret = all_returns['S&P 500 (Large Cap)']
tlt_ret = all_returns['Long-term Treasuries']

rolling_corr = spy_ret.rolling(63).corr(tlt_ret)  # ~3 month window

plt.figure(figsize=(13, 5))
plt.plot(rolling_corr.index, rolling_corr, color='#8e44ad', linewidth=1.5)
plt.axhline(0, color='black', linewidth=0.8)
plt.axhline(rolling_corr.mean(), color='orange', linestyle='--', linewidth=1.5,
            label=f'Average: {rolling_corr.mean():.2f}')
plt.fill_between(rolling_corr.index, rolling_corr, 0,
                 where=(rolling_corr < 0), alpha=0.2, color='green',
                 label='Negative corr (diversification works)')
plt.fill_between(rolling_corr.index, rolling_corr, 0,
                 where=(rolling_corr > 0), alpha=0.2, color='red',
                 label='Positive corr (diversification fails)')

plt.title('Rolling 3-Month Correlation: S&P 500 vs Long-Term Treasuries',
          fontweight='bold')
plt.ylabel('Correlation Coefficient')
plt.ylim(-1, 1)
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()

print("\n⚠️  2022 Warning:")
print("   Notice the correlation turning POSITIVE in 2022 — both stocks AND bonds")
print("   fell simultaneously. The traditional 60/40 portfolio broke down.")
print("   This is why 2022 was so painful for most institutional investors.")

---
## 6. Drawdown Analysis

### Theory

A **drawdown** measures the decline from a portfolio's peak value to its subsequent trough:

$$
DD_t = \frac{V_t - \max_{s \leq t}(V_s)}{\max_{s \leq t}(V_s)}
$$

The **Maximum Drawdown (MDD)** is the largest peak-to-trough decline over the entire period:

$$
MDD = \min_t(DD_t)
$$

### Why drawdown matters for traders

The CAGR alone is a seductive but incomplete picture. Two portfolios can have the same CAGR but very different drawdown profiles — and the one with the smaller drawdown is almost always **psychologically and practically superior**:

- You need to stay in the trade to realize the return. A -60% drawdown can force panic selling.
- If using leverage, a deep drawdown can trigger a **margin call** before recovery.
- The **Calmar Ratio** = CAGR / |MDD| is a key metric in professional hedge funds.

> 💡 Your notes: *"Los retornos de invertir en el mercado de valores son positivos en el largo plazo"* (stock market returns are positive in the long run) — but you have to *survive* the drawdowns to get there.

In [ ]:
# ── 6a. Compute and plot drawdowns for major asset classes ────────────────────

def compute_drawdown(price_series):
    """Compute the drawdown series from a price series."""
    rolling_max = price_series.cummax()
    drawdown = (price_series - rolling_max) / rolling_max
    return drawdown

# Focus on equity benchmarks for clarity
focus_assets = ['S&P 500 (Large Cap)', 'Russell 2000 (Small Cap)',
                'Nasdaq 100 (Tech)', 'Long-term Treasuries']
focus_colors = ['#2980b9', '#e67e22', '#c0392b', '#27ae60']

# Compute wealth index from start for each asset
wealth = (1 + all_returns[focus_assets].fillna(0)).cumprod()

fig, axes = plt.subplots(2, 1, figsize=(13, 9))

# Wealth chart
for col, color in zip(focus_assets, focus_colors):
    axes[0].plot(wealth.index, wealth[col], label=col, color=color, linewidth=1.8)
axes[0].set_title('Cumulative Wealth ($1 invested)', fontweight='bold')
axes[0].set_ylabel('Portfolio Value')
axes[0].legend(fontsize=9)

# Drawdown chart
for col, color in zip(focus_assets, focus_colors):
    dd = compute_drawdown(wealth[col])
    axes[1].fill_between(dd.index, dd * 100, 0, alpha=0.4, color=color, label=col)
    axes[1].plot(dd.index, dd * 100, color=color, linewidth=0.8, alpha=0.8)

axes[1].set_title('Drawdown from Peak (%)', fontweight='bold')
axes[1].set_ylabel('Drawdown (%)')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── 6b. Summary table with Calmar Ratio ──────────────────────────────────────
risk_table = pd.DataFrame(index=focus_assets)

n_yrs = len(all_returns) / TRADING_DAYS
for asset in focus_assets:
    ret_series = all_returns[asset].dropna()
    wealth_series = (1 + ret_series).cumprod()
    dd_series = compute_drawdown(wealth_series)

    cagr = (wealth_series.iloc[-1]) ** (1 / n_yrs) - 1
    mdd  = dd_series.min()
    ann_vol = ret_series.std() * np.sqrt(TRADING_DAYS)
    sharpe = (cagr - rf_annual) / ann_vol
    calmar = cagr / abs(mdd) if mdd != 0 else np.nan

    risk_table.loc[asset, 'CAGR (%)']       = round(cagr * 100, 2)
    risk_table.loc[asset, 'Volatility (%)'] = round(ann_vol * 100, 2)
    risk_table.loc[asset, 'Max Drawdown (%)'] = round(mdd * 100, 2)
    risk_table.loc[asset, 'Sharpe Ratio']   = round(sharpe, 3)
    risk_table.loc[asset, 'Calmar Ratio']   = round(calmar, 3)

print("Extended Risk-Return Summary (focus assets):")
print(risk_table.to_string())
print("\nCalmar Ratio = CAGR / |Max Drawdown| — higher is better")
print("Sharpe Ratio = (CAGR - Rf) / Volatility — higher is better")

---
## 7. Module Summary & Key Takeaways

| Concept | Key Formula | Trading Application |
|---|---|---|
| Risk-free rate | $r_f$ = 13W T-bill yield | Benchmark for all risk premiums; CAPM input |
| Risk premium | $E[R] - r_f$ | What you earn for bearing risk |
| Sharpe Ratio | $(R_p - r_f) / \sigma_p$ | Compare strategies on risk-adjusted basis |
| Max Drawdown | $\min(V_t / V_{peak} - 1)$ | Measures real-world pain; critical for position sizing |
| Calmar Ratio | CAGR / |MDD| | Reward-per-unit-of-drawdown |
| Correlation | $\rho_{A,B} \in [-1,1]$ | Foundation of diversification |

### 🔑 Three Rules to Carry Into Your Trading

1. **Never evaluate return without risk.** A strategy with a 20% return and 40% drawdown is worse than a 12% return with a 10% drawdown. Always use Sharpe, Calmar, or similar risk-adjusted metrics.

2. **Correlations change — especially in crises.** Assets that normally diversify each other (stocks + bonds) can become correlated when macro conditions shift. The 2022 bear market is a recent reminder.

3. **The risk-free rate is not a constant.** In a ZIRP (zero interest rate policy) world, almost every asset looks good. When the risk-free rate is 5%, the bar rises significantly. Always compare your strategy returns against the *current* risk-free rate.

---

### ➡️ Next: Module 1.3 — Measuring Risk I: Volatility
We'll go deep on volatility — computing it, modeling it over time, and understanding why it's the most important single input in options pricing, position sizing, and risk management.